In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/README.md
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/tokenizer.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/tokenizer_config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/chat_template.jinja
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/model.safetensors
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/processor_config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/generation_config.json
/kaggle/input/competitions/gemma-4-good-hackathon/NOTE.md


!pip install -q -U kaggle-benchmarks

## Democratic Hammurabi Benchmark Task Definition

In [3]:
from hammurabi_env import DemocraticHammurabi

class LLMDemocraticHammurabi:
    """
    A wrapper around our updated DemocraticHammurabi environment for LLMs.
    """
    def __init__(self, max_years=12):
        self.env = DemocraticHammurabi(max_years=max_years)
        self.state = self.env.reset()
        self.last_info = self.env._build_info()

    def play_turn(self, action_land: float, action_civ_procure: float, action_caravan_trade: float, action_feed: float, action_plant: float, action_project: float):
        actions = [
            action_land,
            action_civ_procure,
            action_caravan_trade,
            action_feed,
            action_plant,
            action_project
        ]
        self.state, reward, done, info = self.env.step(actions)
        self.last_info = info
        return done, info.get('reason', '')

    def get_slm_payload(self):
        """The data block passed to the LLM to generate the next turn's story."""
        state = self.state
        year, pop, grain, land, land_price, silver, grain_price, f_app, w_app, e_app, yrs_to_elec, f_pop, w_pop, e_pop, civ_grain, f_silver, e_silver, w_silver, dom_grain_price, caravan_silver = state

        optimal_food = int(pop * 20)

        # Formatting a detailed state report for the LLM
        return f"""
        REPORT:
        Year: {int(year)}
        Years Until Next Election: {int(yrs_to_elec)}

        DEMOGRAPHICS:
        Total Population: {int(pop)}
        - Farmers: {int(f_pop)}
        - Workers: {int(w_pop)}
        - Elites: {int(e_pop)}

        STATE RESOURCES (ROYAL):
        Acres of Land: {int(land)}
        Bushels in Royal Storage: {int(grain)}
        Royal Silver Vault: {int(silver)}

        CIVILIAN RESOURCES (PRIVATE):
        Civilian Grain Storage: {int(civ_grain)}
        Farmer Silver: {int(f_silver)}
        Worker Silver: {int(w_silver)}
        Elite Silver: {int(e_silver)}

        MARKET PRICES:
        Current Land Price: {land_price:.2f} silver/acre
        Foreign Caravan Grain Price: {grain_price:.2f} silver/bushel
        Domestic Grain Price: {dom_grain_price:.2f} silver/bushel
        Foreign Caravan Silver Available: {int(caravan_silver)} silver

        FOOD REQUIREMENT:
        (NOTE: Your {int(pop)} people require {optimal_food} bushels to avoid starvation this year).

        POLITICAL POLLS (Must stay above 50% average to win election):
        Farmers Approval: {f_app:.1f}%
        Workers Approval: {w_app:.1f}%
        Elites Approval: {e_app:.1f}%
        """


You are the Grand Vizier of Democratic Babylon.
I will provide you with a 'REPORT' containing raw data about the kingdom.
1. Think silently about the implications of the data, your budget, and the political polls.
2. Calculate your budget based on the Laws of Babylon.
3. Once you have reasoned through your strategy, you MUST call the `issue_decree` tool to finalize your decisions.
4. You must output your tool call using EXACTLY this syntax, replacing the values with your calculated numbers:
<|tool_call>call:issue_decree{acres_to_buy: [number], bushels_to_feed: [number], acres_to_plant: [number]}<tool_call|>

THE LAWS OF BABYLON (GAME MECHANICS):
Before making your decrees, you MUST calculate your budget in your thought block using these exact rules:
1. FEEDING: 1 person requires exactly 20 bushels to survive the year. If you feed them less, people will starve. Starvation drastically lowers Worker approval and causes instant impeachment if too high!
2. PLANTING: It costs exactly 1 bushe

In [4]:
import kaggle_evaluation.core.benchmarks as kbench

@kbench.task(name="democratic_hammurabi_v2")
def play_democratic_hammurabi(
    llm: kbench.Actor,
    max_years: int = 12,
    max_retries_per_turn: int = 3
) -> None:
    """
    Evaluates an LLM's ability to act as the Grand Vizier of Democratic Babylon,
    balancing resources, monetary policies, and political factions.
    """
    game = LLMDemocraticHammurabi(max_years=max_years)
    current_decree = []

    def issue_decree(
        action_land: float, 
        action_civ_procure: float, 
        action_caravan_trade: float, 
        action_feed: float, 
        action_plant: float, 
        action_project: float
    ) -> str:
        """
        Issues the royal decrees for the year, utilizing continuous floats.
        
        Args:
            action_land (float): -1.0 to 1.0 (negative to sell land, positive to buy land from civilians).
            action_civ_procure (float): 0.0 to 1.0 (fraction of affordable domestic civilian grain to buy for the royal silos).
            action_caravan_trade (float): -1.0 to 1.0 (negative to sell royal grain for silver to caravan, positive to buy foreign grain with royal silver).
            action_feed (float): 0.0 to 1.0 (fraction of royal grain to use to feed workers and then civilians).
            action_plant (float): 0.0 to 1.0 (fraction of remaining royal grain to plant on land).
            action_project (float): 0.0 to 1.0 (0-0.25=None, 0.26-0.50=Canal Dredging, 0.51-0.75=Granary Fortification, 0.76-1.0=Land Reclamation).
        """
        
        # Clamp inputs just to be safe
        action_land = max(-1.0, min(1.0, float(action_land)))
        action_civ_procure = max(0.0, min(1.0, float(action_civ_procure)))
        action_caravan_trade = max(-1.0, min(1.0, float(action_caravan_trade)))
        action_feed = max(0.0, min(1.0, float(action_feed)))
        action_plant = max(0.0, min(1.0, float(action_plant)))
        action_project = max(0.0, min(1.0, float(action_project)))

        current_decree.clear()
        current_decree.extend([action_land, action_civ_procure, action_caravan_trade, action_feed, action_plant, action_project])
        return "The decrees have been recorded."

    system_prompt = """You are the Grand Vizier of Democratic Babylon.
I will provide you with a 'REPORT' containing raw data about the kingdom.
1. Think silently about the implications of the data, the dual-market silver economy, and the political polls.
2. Determine your optimal strategy based on the Laws of Babylon.
3. Once you have reasoned through your strategy, you MUST call the `issue_decree` tool to finalize your decisions.

THE LAWS OF BABYLON (GAME MECHANICS):
You must balance monetary capital (silver) and biological capital (grain) over a 12-year period.
Your decrees consist of 6 floats:
1. `action_land` (-1.0 to 1.0): Positive to buy land from civilians using royal silver. Negative to sell royal land to civilians for silver.
2. `action_civ_procure` (0.0 to 1.0): Buy domestic civilian grain for royal silos. Fraction of what you can afford.
3. `action_caravan_trade` (-1.0 to 1.0): Positive to buy foreign grain with silver. Negative to sell royal grain to the caravan for silver.
4. `action_feed` (0.0 to 1.0): Fraction of current royal grain used to feed state workers (and then civilians). 1 person needs 20 bushels. Starvation leads to impeachment!
5. `action_plant` (0.0 to 1.0): Fraction of REMAINING royal grain to plant as seed. 1 bushel plants 1 acre.
6. `action_project` (0.0 to 1.0): Public Works. 
   - 0.00-0.25: None
   - 0.26-0.50: Canal Dredging (costs 500 silver, >= 30 workers, boosts yield)
   - 0.51-0.75: Granary Fortification (costs 400 silver, >= 20 workers, prevents rats)
   - 0.76-1.00: Land Reclamation (costs 600 silver, >= 35 workers, adds land)

THE RULES OF POLITICS (HOW TO WIN):
1. ELECTIONS: An election occurs every 4 years. If your average approval drops below 50%, you will be impeached and lose the game.
2. FARMERS: Love when you buy land and plant seeds.
3. WORKERS: Love extra food and public projects. HATE starvation.
4. ELITES: Care about total kingdom wealth and high silver.

You must issue your decree with 6 float values to balance the factions and survive!"""
    
    kbench.system.send(system_prompt)
    
    done = False
    reason = ""
    
    while not done:
        report = "Current Kingdom Status:\n" + game.get_slm_payload()
        
        current_decree.clear()
        retries = 0
        prompt_text = report
        
        while len(current_decree) == 0:
            if retries >= max_retries_per_turn:
                done = True
                reason = "Impeached for analysis paralysis (failed to issue a valid decree after max retries)."
                break
                
            try:
                response = llm.prompt(prompt_text, tools=[issue_decree])
            except Exception as e:
                done = True
                reason = f"Impeached for analysis paralysis (LLM Error: {type(e).__name__})."
                break
            
            if len(current_decree) == 0:
                last_msg = kbench.chats.current().messages[-1]
                calls = getattr(last_msg, "tool_calls", [])
                
                if not calls and last_msg.sender != "tool":
                    prompt_text = "You did not issue a decree. You MUST call the `issue_decree` tool with 6 float values to end your turn."
                else:
                    prompt_text = "Please review the Accountant's error and recalculate your decree."
                retries += 1
            else:
                break
                
        if len(current_decree) == 6:
            done, reason = game.play_turn(*current_decree)
                
    survived = (game.env.year > max_years)
    final_score = game.env._calculate_reward()
    
    # Collect detailed final stats similar to the user's report
    stats_msg = (
        f"Game Over Reason: {reason or game.env.game_over_reason} | "
        f"Final Pop: {game.env.population} | "
        f"Final Royal Vault: {int(game.env.silver)} | "
        f"Final Private M2: {int(game.env.civilian_silver)} | "
        f"Final Royal Silos: {game.env.grain} | "
        f"Final Civ Granary: {game.env.civilian_grain} | "
        f"Final Land: {game.env.land} | "
        f"Final Score: {final_score:.1f}"
    )
    
    kbench.assertions.assert_true(
        survived,
        expectation=f"Agent must survive 12 years. | {stats_msg}"
    )


In [ ]:

# Run the benchmark
result = play_democratic_hammurabi.run(
    llm=kbench.llm,
    max_years=12
)

print("\n--- Benchmark Results ---")
print(f"Status: {result.status.name}")
if result.assertion_results:
    assertion = result.assertion_results[0]
    print(f"Passed: {assertion.passed}")
    print(f"Details: {assertion.expectation}")
